# ⚡ التنبؤ بالطلب على الطاقة باستخدام ML
**Energy Demand Forecasting with Machine Learning**

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "ml/b8_demand_forecast"          # مسار المشروع داخل portfolio/
PKGS    = ["xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| الموضوع | المصدر المقترح |
|---|---|
| Time Series Feature Engineering (lags, rolling) | Hyndman Ch.5 / Géron Ch.15 |
| XGBoost / Gradient Boosting | Géron Ch.7 |
| Time Series Train/Test Split | Hyndman Ch.3 (لا تستخدم random split!) |
| Evaluation: RMSE, MAE, MAPE | Hyndman Ch.5 |

## 🎯 بيُستخدم في إيه (استخدام واقعي)
- **شركات الكهرباء** بتتنبأ بالطلب عشان تخطط الإنتاج وتتجنب الانقطاعات.
- **المصانع** بتستخدم forecasting عشان تحدد كمية المواد الخام المطلوبة.
- **الـ retail** بيتنبأ بالطلب عشان يدير المخزون ويمنع الـ out-of-stock.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

## 1️⃣ تحميل البيانات والاستكشاف

In [ ]:
# TODO: حمّل البيانات وحوّل timestamp لـ datetime ورتّب بالتاريخ
df = pd.read_csv("data/energy_consumption_data.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print("Shape:", df.shape)
df.head()

In [ ]:
# TODO: ارسم 3 رسومات time series (consumption, temperature, humidity)
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
# أكمل...
plt.tight_layout()
plt.show()

## 2️⃣ هندسة الميزات الزمنية (Time Series Features)
**القاعدة الذهبية:** في السلاسل الزمنية، الميزات الأهم هي:
1. **Lag features** — قيم الماضي
2. **Rolling statistics** — متوسط/انحراف آخر N ساعة
3. **Calendar features** — الساعة، اليوم، الشهر

In [ ]:
# TODO: أضف ميزات زمنية:
# 1) Calendar: hour, dayofweek, month, is_weekend
# 2) Lag features: lag_1, lag_2, ..., lag_24
# 3) Rolling: rolling_mean, rolling_std بأحجام نوافذ مختلفة
# ⚠️ مهم: shift(1) قبل الـ rolling عشان ما نسرّبش بيانات المستقبل!
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = _____

for lag in [1, 2, 3, 6, 12, 24]:
    df[f"lag_{lag}"] = _____

# أكمل...
df = df.dropna().reset_index(drop=True)
print(f"Shape after features: {df.shape}")

## 3️⃣ تقسيم البيانات (Temporal Split)
**⚠️ مهم جداً:** في السلاسل الزمنية، **ما بنستخدمش** `train_test_split` العشوائي!
بنقسّم **بالترتيب الزمني** — الأقدم للتدريب والأحدث للاختبار.

In [ ]:
# TODO: قسّم البيانات 80/20 بالترتيب الزمني (مش عشوائي!)
# train = أول 80%، test = آخر 20%
target = "consumption_kwh"
features = [c for c in df.columns if c not in [target, "timestamp"]]

split_idx = int(len(df) * 0.8)
train = _____
test = _____
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]
print(f"Train: {len(train)}, Test: {len(test)}")

## 4️⃣ تدريب ومقارنة النماذج

In [ ]:
# TODO: درّب وقارن 3 نماذج: Ridge, Random Forest, XGBoost
# قيّم بـ RMSE, MAE, R², MAPE
models = {
    "Ridge": Ridge(alpha=1),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                                 random_state=42, verbosity=0),
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    # أكمل التقييم...


## 5️⃣ تصوير التنبؤات

In [ ]:
# TODO: ارسم خط الطلب الفعلي وخط التنبؤ من أفضل نموذج
fig, ax = plt.subplots(figsize=(14, 5))
# أكمل...
plt.show()

## 6️⃣ أهمية الميزات

In [ ]:
# TODO: اعرض أهم 15 ميزة من أفضل نموذج
# أكمل...


## 7️⃣ التحقق بـ Time Series Cross-Validation
بنستخدم **TimeSeriesSplit** عشان نتأكد إن الأداء مش بالصدفة.

In [ ]:
# TODO: استخدم TimeSeriesSplit (5 folds) لتقييم أفضل نموذج
from sklearn.model_selection import cross_val_score
tscv = TimeSeriesSplit(n_splits=5)
# أكمل...


## 8️⃣ الخلاصة

In [ ]:
# TODO: اطبع ملخص نهائي بالنتائج والملاحظات
print("ملخص التنبؤ بالطلب:")
# أكمل...
